<a href="https://colab.research.google.com/github/marcochen819/ETF-SUMMARY/blob/main/20260421.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import requests
import pandas as pd
import time

def get_twse_etf_data(date_str):
    # 網址參數 selectType=0099P 為 ETF 類別
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?response=json&date={date_str}&selectType=0099P"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        resp = requests.get(url, headers=headers)
        data = resp.json()

        if data.get("stat") == "OK":
            # 建立 DataFrame
            df = pd.DataFrame(data["data"], columns=data["fields"])

            # 依需求篩選欄位 (注意：證交所原始欄位名稱必須完全一致)
            selected_cols = [
                "證券代號",
                "證券名稱",
                "外陸資買賣超股數(不含外資自營商)",
                "投信買賣超股數",
                "自營商買賣超股數",
                "三大法人買賣超股數"
            ]

            df = df[selected_cols]
            df.insert(0, '日期', date_str) # 在第一欄補上日期方便識別
            return df
        else:
            return None
    except:
        return None

from datetime import datetime, timedelta

# 1. 設定開始與結束日期
start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 1, 7)  # 之後可以直接改為 datetime.now() 抓到今天

# 2. 自動產生日期清單 (格式: YYYYMMDD)
dates = []
current_date = start_date
while current_date <= end_date:
    dates.append(current_date.strftime("%Y%m%d"))
    current_date += timedelta(days=1)

# 接下來就接你原本的 for d in dates 迴圈...
print(f"準備抓取的日期範圍：{dates}")

for d in dates:
    # 檢查是否為週末 (5 是週六, 6 是週日)
    date_obj = datetime.strptime(d, "%Y%m%d")
    if date_obj.weekday() >= 5:
        # print(f"{d} 是週末，跳過")
        continue

    print(f"正在抓取 {d}...")
    res = get_twse_etf_data(d)

    if res is not None:
        all_dfs.append(res)
        time.sleep(3) # 只有真的有發出請求時才停頓

# 合併輸出
if all_dfs:
    final_result = pd.concat(all_dfs, ignore_index=True)
    final_result.to_csv("ETF_Summary.csv", encoding="utf-8", index=False)
    print("\n資料已精簡並存檔成功！")
    print(final_result.head()) # 預覽前五筆

準備抓取的日期範圍：['20260101', '20260102', '20260103', '20260104', '20260105', '20260106', '20260107']
正在抓取 20260101...
正在抓取 20260102...
正在抓取 20260105...
正在抓取 20260106...
正在抓取 20260107...

資料已精簡並存檔成功！
         日期    證券代號       證券名稱 外陸資買賣超股數(不含外資自營商)  投信買賣超股數    自營商買賣超股數  \
0  20260102   00878  國泰永續高股息           8,349,228        0  52,891,230   
1  20260102   00919   群益台灣精選高息        22,600,285        0   6,686,398   
2  20260102   00940   元大台灣價值高息           222,105        0  27,022,394   
3  20260102  00665L  富邦恒生國企正2          2,778,000        0  22,803,196   
4  20260102   00900  富邦特選高股息30         6,592,070  200,000  17,527,284   

    三大法人買賣超股數  
0  61,240,458  
1  29,286,683  
2  27,244,499  
3  25,581,196  
4  24,319,354  
